<a href="https://colab.research.google.com/github/tsm-mehmetakiftasoz/tsm_makif/blob/main/lotlay%C4%B1c%C4%B1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [152]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from google.colab import files
import os

# --- Helper Functions ---

def duzenle_sayi_sutunu(df, sutun_adi):
    """
    Sayısal değerleri metin olarak tutan ve binlik/ondalık ayracı içeren sütunu dönüştürür.
    - Nokta (.) binlik ayracı olarak kaldırılır
    - Virgül (,) ondalık ayracı olarak noktaya çevrilir
    - Float olarak dönüştürülür
    - Eğer tam sayı ise integer olarak, değilse float olarak kalır
    - Sonuç string olarak döndürülür (görsel temizlik için)

    Parametreler:
    df : pd.DataFrame
    sutun_adi : str

    Dönüş:
    df (işlenmiş hali)
    """
    df[sutun_adi] = (
        df[sutun_adi]
        .astype(str)
        .str.replace(" ", "", regex=False)
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
        .astype(float)
        .apply(lambda x: str(int(x)) if x.is_integer() else str(x))
    )
    return df

# Temizlenecek değerler listesi (şu anki kodda doğrudan kullanılmıyor, ancak önceki context'ten korundu)
temizlenecekler = ["nan", "Nan", "NaN", ""," ","0","naN","nAn","NAn","nAN"]

def temizle_nan(value):
    if str(value).strip().lower() in [x.lower() for x in temizlenecekler]:
        return pd.NA
    return value

def dolu_mi(x):
    return pd.notna(x) and str(x).strip() != "" and str(x).strip().upper() != "<NA>"

# --- Data Loading ---
df_me2n_2022_2025 = pd.read_csv('/content/me2n_2026_2022.csv', sep=None, engine='python', dtype={"Единица цены":object})

# --- Initial Data Cleaning and Lot Creation ---

# 'Группа материалов' sütunundaki NaN değerleri 'Неопределенные' ile dolduralım
df_me2n_2022_2025['Группа материалов'] = df_me2n_2022_2025['Группа материалов'].fillna('Неопределенные')

# Her bir 'Документ закупки' için benzersiz 'Группа материалов' değerlerini gruplayalım
lot_malzeme_gruplari = df_me2n_2022_2025.groupby('Документ закупки')['Группа материалов'].unique()

# Bu malzeme gruplarını birleştirerek yeni lot isimleri oluşturalım
def get_material_group_lot_names(material_groups_list):
    # Malzeme gruplarının benzersiz ve alfabetik sıraya göre birleştirilmesi
    return ', '.join(sorted(list(set(material_groups_list))))

new_lot_names_by_group = lot_malzeme_gruplari.apply(get_material_group_lot_names)

# new_lot_names_by_group Series'ini DataFrame'e çevirerek birleştirmeye hazırlayalım
lot_names_df = new_lot_names_by_group.reset_index()
lot_names_df.columns = ['Документ закупки', 'Malzeme_Grubu_Lot_Adı']

# df_me2n_2022_2025 DataFrame'ine Malzeme_Grubu_Lot_Adı sütununu ekleyelim
df_me2n_2022_2025 = df_me2n_2022_2025.merge(lot_names_df, on='Документ закупки', how='left')

# Benzersiz Malzeme_Grubu_Lot_Adı değerlerini alalım
unique_lot_names = df_me2n_2022_2025['Malzeme_Grubu_Lot_Adı'].unique()

# 'LOT-X' formatında yeni isimler için bir sözlük oluşturalım
lot_name_mapping = {name: f'LOT-{i+1}' for i, name in enumerate(unique_lot_names)}

# Yeni 'Final_Lot_Adı' sütununu oluşturalım
df_me2n_2022_2025['Final_Lot_Adı'] = df_me2n_2022_2025['Malzeme_Grubu_Lot_Adı'].map(lot_name_mapping)

# Sayısal sütunların temizlenmesi ve dönüşümü (varsa)
numeric_columns_to_clean = [
    'Объем заказа',
    'еще поставить (количество)',
    ' Еще для поставки (стоимость) ',
    'Уже поставлено (количество)',
    'Количество в СЕИ',
    'Единица цены',
    'СтоимЗаказа нетто',
    'Цена нетто'
]

for col in numeric_columns_to_clean:
    if col in df_me2n_2022_2025.columns:
        df_me2n_2022_2025 = duzenle_sayi_sutunu(df_me2n_2022_2025, col)

# Tarih sütunlarının dönüştürülmesi
date_columns = ['Дата документа', 'Дата поставки', 'Дата выставления счета']
for col in date_columns:
    if col in df_me2n_2022_2025.columns:
        df_me2n_2022_2025[col] = pd.to_datetime(df_me2n_2022_2025[col], errors='coerce')

# --- Feature Engineering ---

# 'Материал' sütununu kategorik tipe dönüştürelim
df_me2n_2022_2025['Материал'] = df_me2n_2022_2025['Материал'].astype('category')

# 'Группа материалов' sütununu kategorik tipe dönüştürelim
df_me2n_2022_2025['Группа материалов'] = df_me2n_2022_2025['Группа материалов'].astype('category')

# Hedef değişkenimizi (y) 'Final_Lot_Adı' olarak belirleyelim
y = df_me2n_2022_2025['Final_Lot_Adı']

# Özelliklerimizi (X) 'Материал' ve 'Группа материалов' sütunlarından oluşturalım
X = pd.get_dummies(df_me2n_2022_2025[['Материал', 'Группа материалов']], drop_first=True)

# Eğitim verisindeki benzersiz 'Материал' ve 'Группа материалов' değerlerini saklayalım
unseen_materials = df_me2n_2022_2025['Материал'].unique()
unseen_material_groups = df_me2n_2022_2025['Группа материалов'].unique()

# --- Train-Test Split ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- Model Training and Evaluation (without SMOTE) ---
# SMOTE olmadan doğrudan RandomForestClassifier modelini eğitelim
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test.astype(int))

print("\n--- Model Performansı (SMOTE Uygulanmamış) ---")
print(f'Accuracy: {accuracy_score(y_test, y_pred)}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, zero_division=0))

# --- Prediction on New Data ---

# Yeni veri dosyasının yolu
new_data_filename = '/content/ekran.csv'

try:
    df_new_data = pd.read_csv(new_data_filename, sep=';', engine='python', dtype={"Единица цены":object})
    print(f"Yeni veri '{new_data_filename}' başarıyla yüklendi.")
except Exception as e:
    print(f"Hata oluştu: {e}")
    print("Lütfen dosya adını, ayırıcıyı veya veri tiplerini kontrol edin.")
    # Hata durumunda boş bir DataFrame oluşturup çıkış yapalım
    df_new_data = pd.DataFrame(columns=['Материал', 'Группа материалов', 'Tahmin_Edilen_Lot_Adı'])

if not df_new_data.empty:
    # Orijinal yeni veriyi kopyalayalım ki tahminleri ve 'LOT-0' atamalarını üzerinde yapabilelim
    df_new_data_processed = df_new_data.copy()

    # Apply the same preprocessing steps to the new data
    df_new_data_processed['Группа материалов'] = df_new_data_processed['Группа материалов'].fillna('Неопределенные')
    df_new_data_processed['Материал'] = df_new_data_processed['Материал'].astype('category')
    df_new_data_processed['Группа материалов'] = df_new_data_processed['Группа материалов'].astype('category')

    # One-hot encoding for new data
    X_new_data = pd.get_dummies(df_new_data_processed[['Материал', 'Группа материалов']], drop_first=True)

    # Align columns of X_new_data with X_train (model's training features)
    # Add missing columns from X_train to X_new_data and fill with False (0)
    missing_cols_in_new = set(X_train.columns) - set(X_new_data.columns)
    for c in missing_cols_in_new:
        X_new_data[c] = False

    # Drop extra columns in X_new_data that are not in X_train
    extra_cols_in_new = set(X_new_data.columns) - set(X_train.columns)
    X_new_data = X_new_data.drop(columns=list(extra_cols_in_new))

    # Ensure the order of columns is the same as in X_train
    X_new_data = X_new_data[X_train.columns]

    # İlk tahminleri yapalım
    y_new_pred_temp = model.predict(X_new_data.astype(int))
    df_new_data_processed['Tahmin_Edilen_Lot_Adı'] = y_new_pred_temp

    # 'LOT-0' ataması için kontrol edelim:
    # Her bir yeni veri satırındaki 'Материал' veya 'Группа материалов' eğitim setinde hiç görüldü mü?
    for index, row in df_new_data_processed.iterrows():
        material_val = row['Материал']
        group_material_val = row['Группа материалов']

        # Eğer Malzeme veya Malzeme Grubu eğitim setinde hiç görülmediyse 'LOT-0' ata
        if material_val not in unseen_materials or group_material_val not in unseen_material_groups:
            df_new_data_processed.loc[index, 'Tahmin_Edilen_Lot_Adı'] = 'LOT-0'

    print("\n--- Yeni Veri Seti ve Tahmin Edilen Lot Adları (İlk 10 Satır) ---")
    display(df_new_data_processed[['Материал', 'Группа материалов', 'Tahmin_Edilen_Lot_Adı']].head(10))

    # Tahmin edilen lot adlarını içeren yeni veriyi Excel'e kaydedelim
    output_excel_filename = 'new_data_with_predicted_lots_no_smote.xlsx'
    df_new_data_processed.to_excel(output_excel_filename, index=False)

    print(f"Yeni veriler '{output_excel_filename}' olarak kaydedildi.")

    # Excel dosyasını indirme bağlantısı sağlayalım
    files.download(output_excel_filename)
else:
    print("Yeni veri yüklenemediği için tahmin yapılamadı ve Excel dosyası oluşturulamadı.")



--- Model Performansı (SMOTE Uygulanmamış) ---
Accuracy: 0.7413793103448276

Classification Report:
              precision    recall  f1-score   support

       LOT-1       0.00      0.00      0.00         0
      LOT-11       0.00      0.00      0.00         1
      LOT-12       1.00      1.00      1.00         2
      LOT-13       0.00      0.00      0.00         1
      LOT-14       0.00      0.00      0.00         2
      LOT-16       1.00      1.00      1.00        19
      LOT-18       1.00      1.00      1.00         9
      LOT-19       0.00      0.00      0.00         1
      LOT-23       1.00      1.00      1.00         2
      LOT-24       0.00      0.00      0.00         1
      LOT-25       0.25      0.25      0.25         4
      LOT-27       0.50      1.00      0.67         3
      LOT-28       1.00      1.00      1.00         1
      LOT-31       1.00      1.00      1.00         4
       LOT-4       0.00      0.00      0.00         4
       LOT-5       0.00      0.00 

/tmp/ipykernel_38247/3784502564.py:168: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_new_data[c] = False
/tmp/ipykernel_38247/3784502564.py:168: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_new_data[c] = False
/tmp/ipykernel_38247/3784502564.py:168: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
 


--- Yeni Veri Seti ve Tahmin Edilen Lot Adları (İlk 10 Satır) ---


,Материал,Группа материалов,Tahmin_Edilen_Lot_Adı
0,4.500091e+09,MZ-2017,LOT-0
1,4.500089e+09,MZ-2032,LOT-0
2,4.500149e+09,MZ-2032,LOT-0
3,4.500149e+09,MZ-2032,LOT-0
4,4.500093e+09,MZ-2032,LOT-0
5,4.500087e+09,MZ-6017,LOT-0
6,4.500142e+09,MZ-2029,LOT-0
7,4.500074e+09,MZ-6004,LOT-0
8,4.500207e+09,MZ-6008,LOT-0
9,4.500081e+09,MZ-2068,LOT-0


Yeni veriler 'new_data_with_predicted_lots_no_smote.xlsx' olarak kaydedildi.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>